# Time-Series Variability Demo — analiza rzeczywistych bloków

Notebook analizuje najnowszy dostępny blok `machine` i/lub `pv`. Jeśli wcześniejsza sesja zakończyła się po pierwszym bloku, notebook nadal pokaże analizę tego, co faktycznie zostało zebrane, zamiast kończyć się błędem.

In [ ]:
from pathlib import Path
import ast, json, math, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

def find_demo_root():
    start = Path.cwd().resolve()
    candidates = [start, start / "tobii-pytracker-demo", *start.parents]
    for c in candidates:
        if c.name == "tobii-pytracker-demo" and (c / "examples").is_dir():
            return c
        nested = c / "tobii-pytracker-demo"
        if nested.is_dir() and (nested / "examples").is_dir():
            return nested.resolve()
    raise FileNotFoundError("Cannot locate tobii-pytracker-demo from current working directory")

DEMO_ROOT = find_demo_root()
print(f"DEMO_ROOT={DEMO_ROOT}")

def newest_session(root: Path):
    sessions = [p for p in root.iterdir() if p.is_dir() and (p / "data.csv").is_file()] if root.is_dir() else []
    if not sessions:
        raise FileNotFoundError(f"No session with data.csv under {root}")
    return max(sessions, key=lambda p: (p / "data.csv").stat().st_mtime)

def parse_struct(value, expected_type, default):
    if isinstance(value, expected_type):
        return value
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return default
    text = str(value).strip()
    if not text or text.lower() == "nan":
        return default
    for parser in (json.loads, ast.literal_eval):
        try:
            parsed = parser(text)
            if isinstance(parsed, expected_type):
                return parsed
        except Exception:
            pass
    return default

def first_value(d, *names):
    for name in names:
        if name in d and d[name] is not None:
            return d[name]
    return None

def flatten_gaze(raw, set_name):
    records=[]
    for slide_index, row in raw.reset_index(drop=True).iterrows():
        gaze=parse_struct(row.get("gaze_data"), list, [])
        for sample in gaze:
            if not isinstance(sample, dict):
                continue
            x=first_value(sample, "avg_gaze_x", "gaze_x", "x")
            y=first_value(sample, "avg_gaze_y", "gaze_y", "y")
            if x is None:
                lx,rx=sample.get("gaze_x_left"),sample.get("gaze_x_right")
                x=(lx+rx)/2 if lx is not None and rx is not None else (lx if lx is not None else rx)
            if y is None:
                ly,ry=sample.get("gaze_y_left"),sample.get("gaze_y_right")
                y=(ly+ry)/2 if ly is not None and ry is not None else (ly if ly is not None else ry)
            t=first_value(sample, "system_time", "time", "timestamp", "logged_time")
            if x is None or y is None:
                continue
            records.append({"set_name":set_name,"slide_index":int(slide_index),"input_data":row.get("input_data"),
                            "classification":str(row.get("classification","")).lower(),
                            "avg_gaze_x":float(x),"avg_gaze_y":float(y),
                            "system_time":float(t) if t is not None else float(len(records))})
    return pd.DataFrame.from_records(records)

def stimulus_id_from_screenshot(value):
    name=Path(str(value).replace("\\","/")).name
    return name[:-4].lower() if name.lower().endswith(".png") else name.lower()

def parse_timeseries_bboxes(value):
    objects=parse_struct(value,dict,{})
    items=objects.get("timeseries_bboxes",[]) if isinstance(objects,dict) else []
    out=[]
    for item in items:
        try:
            b=item["bbox"]; out.append({"start_idx":int(item["start_idx"]),"end_idx":int(item["end_idx"]),"cx":float(b["cx"]),"w":float(b["w"])})
        except Exception: pass
    return sorted(out,key=lambda x:(x["start_idx"],x["cx"]))

def sample_index_for_x(x,boxes):
    if not boxes: return None
    for b in boxes:
        if b["cx"]-b["w"]/2 <= x <= b["cx"]+b["w"]/2: return b["start_idx"]
    return min(boxes,key=lambda b:abs(b["cx"]-x))["start_idx"]


In [ ]:
from tobii_pytracker.analyze import FixationAnalyzer
OUTPUT_ROOT=DEMO_ROOT/"output"/"tobii_timeseries_noise_demo"
sessions={}
for domain in ("machine","pv"):
    try: sessions[domain]=newest_session(OUTPUT_ROOT/domain)
    except FileNotFoundError: print(f"WARNING: no completed {domain} block found")
if not sessions: raise FileNotFoundError(f"No time-series sessions under {OUTPUT_ROOT}")
raw_by={}; flat_parts=[]
for domain,session in sessions.items():
    raw=pd.read_csv(session/"data.csv",sep=";");
    if len(raw)!=9: raise RuntimeError(f"{domain}: expected 9 trials, got {len(raw)}")
    raw_by[domain]=raw; flat_parts.append(flatten_gaze(raw,f"{domain}:{session.name}"))
flat=pd.concat([x for x in flat_parts if not x.empty],ignore_index=True) if any(not x.empty for x in flat_parts) else pd.DataFrame()
analysis_dir=OUTPUT_ROOT/"analysis_latest"; analysis_dir.mkdir(parents=True,exist_ok=True); flat.to_csv(analysis_dir/"flattened_gaze.csv",index=False)
fixations=FixationAnalyzer(analysis_dir,method="dispersion").analyze(flat) if not flat.empty else pd.DataFrame()
if not fixations.empty: fixations.to_csv(analysis_dir/"fixations.csv",index=False)


In [ ]:
rows=[]
for domain,raw in raw_by.items():
    set_name=f"{domain}:{sessions[domain].name}"
    for slide_index,trial in raw.reset_index(drop=True).iterrows():
        stim=stimulus_id_from_screenshot(trial.get("screenshot_file","")); parts=stim.split("_"); noise_class=parts[1] if len(parts)>=3 else str(trial.get("classification","")).lower()
        gaze=flat[(flat["set_name"]==set_name)&(flat["slide_index"]==slide_index)] if not flat.empty else pd.DataFrame()
        fx=fixations[(fixations["set_name"]==set_name)&(fixations["slide_index"]==slide_index)] if not fixations.empty else pd.DataFrame()
        boxes=parse_timeseries_bboxes(trial.get("objects_bboxes")); visited=set()
        for _,f in fx.iterrows():
            idx=sample_index_for_x(float(f["x_mean"]),boxes)
            if idx is not None: visited.add(min(7,int(idx*8/max(len(boxes),1))))
        expected=str(trial.get("classification","")).lower(); response=str(trial.get("user_classification","")).lower()
        rows.append({"domain":domain,"session":sessions[domain].name,"slide_index":slide_index,"stimulus_id":stim,"noise_class":noise_class,"expected":expected,"response":response,"correct":expected==response,"gaze_samples":len(gaze),"fixation_count":len(fx),"fixation_dwell_s":float(pd.to_numeric(fx.get("duration"),errors="coerce").fillna(0).sum()) if not fx.empty else 0.0,"time_axis_coverage":len(visited)/8,"timeseries_bbox_count":len(boxes)})
metrics=pd.DataFrame(rows); metrics.to_csv(analysis_dir/"trial_metrics.csv",index=False)
summary=metrics.groupby(["domain","noise_class"],as_index=False).agg(trials=("slide_index","size"),accuracy=("correct","mean"),gaze_available_rate=("gaze_samples",lambda s:(s>0).mean()),mean_fixations=("fixation_count","mean"),mean_dwell_s=("fixation_dwell_s","mean"),mean_time_axis_coverage=("time_axis_coverage","mean"))
summary.to_csv(analysis_dir/"domain_class_summary.csv",index=False); display(summary)
summary.pivot(index="noise_class",columns="domain",values="accuracy").plot(kind="bar",ylim=(0,1),title="Accuracy by domain and variability"); plt.show()
print(f"blocks={list(sessions)}; gaze_missing_trials={(metrics.gaze_samples==0).sum()}; analysis_dir={analysis_dir}")
print("NATIVE_TIMESERIES_ANALYSIS_PASS")
